<a href="https://colab.research.google.com/github/Zeshan811/StarterNotebookA1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub

import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

Paste your Hugging Face READ token (hf_...): ··········
dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [3]:
schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df()
print(schema[['column_name', 'column_type']].to_string())

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

Unit of analysis (grain): one row = one content item, for one client, on one day —
i.e. content_hash_id + client_hash_id + report_date in fact_content_daily_performance.

Time window: I will iterate on a mid-panel month, 2026-03, per the assignment's warning
— the _sample table is the sealed final month (June 2026) and is only for testing query
mechanics, never for developing label logic.

Table(s) I'll use: fact_content_daily_performance (daily metrics), joined to dim_content
(content metadata) on content_hash_id where needed.

What I'd predict or rank: Nothing — my lane (clustering) is unsupervised. Instead of a
label, I will build a small feature frame per content item (aggregated over the month)
to feed a future clustering step. For the leakage exercise below, I will temporarily
build a toy label just to demonstrate the trap, then remove it — it is not part of my
real contract.

One thing I deliberately exclude: any FlyRank product decision output — health_score,
priority_score, action_type, or refresh/review flags. These are not observable signals;
they are the product's own conclusions, and including them would mean my clustering just
reproduces an existing rule instead of discovering structure from raw evidence.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| gsc_impressions | Feature | Observed demand/visibility signal, known at end of day |
| gsc_clicks | Feature | Observed engagement signal, known at end of day |
| gsc_avg_position | Feature | Observed ranking signal, known at end of day |
| ga4_sessions | Feature | Observed analytics signal, known at end of day |
| ga4_engaged_sessions / ga4_total_engagement_sec | Feature | Observed engagement depth signal, known at end of day |
| content_hash_id, client_hash_id | Context (join keys) | Used only to join/group, carry no signal themselves |
| report_date, month | Context | Defines the time window, not a feature or label |
| ga4_data_available | Context (filter) | Used to check availability with IS TRUE, not fed into clustering |
| toy "grew next month" flag | Label (leakage demo only) | Built on purpose, temporarily, only to prove the leakage trap — then deleted |
| health_score, priority_score, action_type | Excluded | Not in the release at all, and even if rebuilt, these are the product's own decisions — using them would let a model copy an existing rule instead of finding real signal |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_row_count,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_combo_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()

print(grain_check)
print("\nGrain confirmed:", grain_check['total_row_count'][0] == grain_check['distinct_combo_count'][0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_row_count  distinct_combo_count
0          9841378               9841378

Grain confirmed: True


In [5]:
count_and_span = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()

print(count_and_span)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count earliest_date latest_date  client_count  content_count
0    9841378    2026-03-01  2026-03-31            55         331437


In [6]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_with_ga4
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()

print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  rows_with_ga4  pct_with_ga4
0     9841378       413966.0           4.2


Building a feature frame per content item, aggregated over March 2026. Since only 4.2%
of rows have GA4 data available (checked above), I'm using GSC-based features only —
these are available for effectively all rows, avoiding a sparse-data problem.

In [7]:
feature_frame = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position_month,
        (SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)) AS ctr_month,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    GROUP BY content_hash_id
""").df()

print(feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 6)


,content_hash_id,total_impressions,total_clicks,avg_position_month,ctr_month,days_with_impressions
0,content_39d7361b4945d504,77.0,0.0,4.074107,0.000000,24
1,content_cec711b02f3bbde6,602.0,4.0,4.428747,0.006645,29
2,content_275b6f7f733016d4,810.0,1.0,4.866123,0.001235,29
3,content_ceaec531566ffcfc,82.0,0.0,8.978086,0.000000,27
4,content_755d951187fcd70a,1858.0,6.0,1.854929,0.003229,30


**Available when? — one line per feature:**

1. total_impressions — knowable at month-end, since GSC impressions are logged daily.
2. total_clicks — knowable at month-end, since GSC clicks are logged daily.
3. avg_position_month — knowable at month-end, since daily position snapshots are recorded as they happen.
4. ctr_month — derived directly from impressions and clicks, so knowable at the same moment.
5. days_with_impressions — knowable at month-end; counts how many days in March the page had any visibility at all.

### The trap: a deliberate leak

To perform the notebook-02 leakage lesson on real warehouse data, I build a toy label
just for this demo — "did this page's impressions grow in the next month (April 2026)
vs March?" — then show what happens if a future-window metric sneaks in as a feature.

In [8]:
next_month = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS next_month_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-04-01' AND report_date < DATE '2026-05-01'
    GROUP BY content_hash_id
""").df()

demo = feature_frame.merge(next_month, on="content_hash_id", how="inner")
demo["grew_next_month"] = (demo["next_month_impressions"] > demo["total_impressions"]).astype(int)

print("Toy label built. Positive rate:", demo["grew_next_month"].mean().round(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Toy label built. Positive rate: 0.24


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

honest_features = ["total_impressions", "total_clicks", "avg_position_month", "ctr_month", "days_with_impressions"]
X_honest = demo[honest_features].fillna(0)
y = demo["grew_next_month"]

honest_model = LogisticRegression(max_iter=1000).fit(X_honest, y)
honest_auc = roc_auc_score(y, honest_model.predict_proba(X_honest)[:, 1])
print(f"HONEST model (features only from March): ROC-AUC = {honest_auc:.3f}")

# Sneak in the leak: a column computed directly from the label's own future window
demo["LEAK_next_month_impressions"] = demo["next_month_impressions"]

leaky_features = honest_features + ["LEAK_next_month_impressions"]
X_leaky = demo[leaky_features].fillna(0)

leaky_model = LogisticRegression(max_iter=1000).fit(X_leaky, y)
leaky_auc = roc_auc_score(y, leaky_model.predict_proba(X_leaky)[:, 1])
print(f"LEAKY model (includes next month's own impressions): ROC-AUC = {leaky_auc:.3f}  <- looks amazing, but it's cheating")

HONEST model (features only from March): ROC-AUC = 0.661
LEAKY model (includes next month's own impressions): ROC-AUC = 1.000  <- looks amazing, but it's cheating


In [10]:
demo = demo.drop(columns=["LEAK_next_month_impressions"])

print("Leaked column removed. Columns remaining:", list(demo.columns))
print(f"\nThe number I keep and report is the HONEST ROC-AUC: {honest_auc:.3f}")
print("The leaky 'near-perfect' score is discarded — it only looked good because the")
print("model was literally given next month's own impressions to compare against itself.")

Leaked column removed. Columns remaining: ['content_hash_id', 'total_impressions', 'total_clicks', 'avg_position_month', 'ctr_month', 'days_with_impressions', 'next_month_impressions', 'grew_next_month']

The number I keep and report is the HONEST ROC-AUC: 0.661
The leaky 'near-perfect' score is discarded — it only looked good because the
model was literally given next month's own impressions to compare against itself.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

- Unbalanced panel: clients have very different amounts of history — check dim_clients
  gsc_data_start/ga4_data_start before trusting any window across all clients.
- GA4 sparsity: only 4.2% of March 2026 rows have GA4 data available, so engagement/session
  signals cannot be used reliably as features at this grain — this is why my 5 features
  are GSC-based only.
- This slice cannot prove causation: even a strong clustering pattern only shows
  association — it cannot say a refresh caused a page to move between archetypes.
- One-month window is a snapshot, not a trend: March 2026 alone can't distinguish a real
  shift from ordinary noise or seasonality.
- Named limitation for my lane: aggregating to one row per content item per month loses
  all day-to-day variation — two pages with identical monthly totals but very different
  daily volatility would look the same to my clustering.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.